In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;

DimCustomer (SCD Type 2)

In [0]:
%sql
CREATE TABLE gold.dim_customer AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY CustomerID) AS CustomerSK,
    CustomerID,
    INITCAP(CustomerName) AS CustomerName,
    LOWER(Email) AS Email,
    TRIM(City) AS City,
    TRIM(Address) AS Address,
    CURRENT_DATE() AS StartDate,
    CAST(NULL AS DATE) AS EndDate,
    1 AS IsActive
FROM silver.customers_clean;

When new data comes:
Step 1 — Expire old record

In [0]:
%sql
UPDATE gold.dim_customer
SET EndDate = CURRENT_DATE(),
    IsActive = 0
WHERE CustomerID IN (
    SELECT s.CustomerID
    FROM silver.customers_clean s
    JOIN gold.dim_customer d
    ON s.CustomerID = d.CustomerID
    WHERE d.IsActive = 1
      AND (
          s.City <> d.City OR
          s.Address <> d.Address OR
          s.Email <> d.Email
      )
);


Step 2 — Insert new record

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;
INSERT INTO gold.dim_customer
SELECT 
    COALESCE((SELECT MAX(CustomerSK) FROM gold.dim_customer), 0) + ROW_NUMBER() OVER (ORDER BY s.CustomerID) AS CustomerSK,
    s.CustomerID,
    s.CustomerName,
    s.Email,
    s.City,
    s.Address,
    CURRENT_DATE() AS StartDate,
    CAST(NULL AS DATE) AS EndDate,
    1 AS IsActive
FROM silver.customers_clean s
LEFT JOIN gold.dim_customer d
ON s.CustomerID = d.CustomerID AND d.IsActive = 1
WHERE 
    d.CustomerID IS NULL   -- new customer
    OR (
        d.IsActive = 1 AND (
            s.City <> d.City OR
            s.Address <> d.Address OR
            s.Email <> d.Email
        )
    );

DimProduct

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
CREATE OR REPLACE TABLE gold.dim_product AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY ProductID) AS ProductSK,
    ProductID,
    TRIM(ProductName) AS ProductName,
    TRIM(Category) AS Category,
    UnitPrice,
    CURRENT_DATE() AS EffectiveDate
FROM (
    SELECT DISTINCT
        ProductID,
        ProductName,
        Category,
        UnitPrice
    FROM silver.products_clean
);

DimStore

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA silver;
CREATE OR REPLACE TABLE gold.dim_store AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreSK,
    StoreID,
    TRIM(StoreName) AS StoreName,
    TRIM(Region) AS Region
FROM (
    SELECT DISTINCT
        StoreID,
        StoreName,
        Region
    FROM silver.stores_clean
);

FactSales

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fact_sales AS
SELECT 
    s.TransactionID,
    d.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,
    (s.Quantity * p.UnitPrice) AS Amount,
    s.TxnDate
FROM silver.sales_clean s
JOIN gold.dim_customer d
ON s.CustomerID = d.CustomerID AND d.IsActive = 1
JOIN gold.dim_product p
ON s.ProductID = p.ProductID
JOIN gold.dim_store st
ON s.StoreID = st.StoreID;

Incremental Load

In [0]:
%sql
USE CATALOG retail_sales_catalog;
USE SCHEMA gold;

UPDATE dim_customer
SET EndDate = CURRENT_DATE(),
    IsActive = 0
WHERE CustomerID IN (
    SELECT s.CustomerID
    FROM retail_sales_catalog.silver.customers_clean s
    JOIN retail_sales_catalog.gold.dim_customer d
      ON s.CustomerID = d.CustomerID
    WHERE d.IsActive = 1
      AND (
          s.City <> d.City OR
          s.Address <> d.Address OR
          s.Email <> d.Email
      )
);

Insert New / Changed Records

In [0]:
%sql
INSERT INTO dim_customer
SELECT 
    ROW_NUMBER() OVER (ORDER BY s.CustomerID) +
    (SELECT COALESCE(MAX(CustomerSK),0) FROM dim_customer),
    s.CustomerID,
    INITCAP(TRIM(s.CustomerName)),
    s.Email,
    s.City,
    s.Address,
    CURRENT_DATE(),
    CAST(NULL AS DATE),
    1
FROM retail_sales_catalog.silver.customers_clean s
LEFT JOIN retail_sales_catalog.gold.dim_customer d
  ON s.CustomerID = d.CustomerID
 AND d.IsActive = 1
WHERE d.CustomerID IS NULL
   OR (
        s.City <> d.City OR
        s.Address <> d.Address OR
        s.Email <> d.Email
   );

FACT TABLE (INCREMENTAL)

In [0]:
%sql
CREATE OR REPLACE TABLE fact_sales AS
SELECT 
    s.TransactionID,
    d.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,
    (s.Quantity * p.UnitPrice) AS Amount,
    s.TxnDate
FROM retail_sales_catalog.silver.sales_clean s
JOIN retail_sales_catalog.gold.dim_customer d
  ON s.CustomerID = d.CustomerID
 AND d.IsActive = 1
JOIN retail_sales_catalog.gold.dim_product p
  ON s.ProductID = p.ProductID
JOIN retail_sales_catalog.gold.dim_store st
  ON s.StoreID = st.StoreID;

Archival

In [0]:
dbutils.fs.mv(
    "s3://retaill-client/retaill-lakehouse/landing/customers_src_20042026100105.csv",
    "s3://retaill-client/retaill-lakehouse/archive/customers_src_20042026100105.csv"
)